<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/wk12_costsensitive_ND_MOSTLYAI_VARIANCE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U "mostlyai[local]" datasets imbalanced-learn 2>&1 | tail -3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 4.6 MB/s eta 0:00:00


In [2]:
import os, re, json, time, platform, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)

RANDOM_STATE = 42
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

MANIFEST = {
    "run_id": RUN_ID,
    "started": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "random_state": RANDOM_STATE,
    "purpose": "cost-sensitive baselines; MostlyAI variance across runs",
}

RESULTS = []

def log_result(method, protocol, granularity, y_true, y_pred,
               train_s=None, infer_s=None, n_test=None, notes=""):
    """Single point of truth. Every number reported anywhere comes from here.

    zero_division=0 matters: a class the model never predicts would otherwise
    return an undefined F1 and be silently dropped from the macro average,
    which is precisely the class we care about.
    """
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cold_label = -3 if granularity == 7 else -1     # 3-class Cold is coded -1
    cold = f1_score(y_true, y_pred, labels=[cold_label],
                    average="macro", zero_division=0)
    row = {
        "method": method, "protocol": protocol, "granularity": granularity,
        "macro_f1": round(float(macro), 4), "cold_f1": round(float(cold), 4),
        "train_s": round(train_s, 1) if train_s else None,
        "infer_ms_per_row": round(1000 * infer_s / n_test, 4)
                            if infer_s and n_test else None,
        "run_id": RUN_ID, "notes": notes,
    }
    RESULTS.append(row)
    print(f"  {method:<38} {protocol:<10} {granularity}-class  "
          f"macro_f1={macro:.4f}  cold_f1={cold:.4f}")
    return row

print("Run ID:", RUN_ID)



Run ID: 20260805_150957


In [3]:
from datasets import load_dataset

dataset  = load_dataset("kopetri/AutoTherm", "indoor")
train_df = dataset["train"].to_pandas()

def extract_participant_id(filename):
    m = re.search(r"participant_\d+", filename)
    return m.group() if m else "unknown"

train_df["participant_id"] = train_df["file_name"].apply(extract_participant_id)
train_df["Label_3class"]   = train_df["Label"].apply(
    lambda x: -1 if x <= -2 else (0 if x <= 1 else 1)
)

TEST_PARTICIPANTS = ["participant_14", "participant_16", "participant_20"]
train_split = train_df[~train_df["participant_id"].isin(TEST_PARTICIPANTS)]
test_split  = train_df[ train_df["participant_id"].isin(TEST_PARTICIPANTS)]

MANIFEST.update({
    "dataset": "kopetri/AutoTherm indoor",
    "test_participants": TEST_PARTICIPANTS,
    "n_train_rows": int(len(train_split)),
    "n_test_rows": int(len(test_split)),
})

print(f"Train: {len(train_split):,} rows, "
      f"{train_split['participant_id'].nunique()} participants")
print(f"Test:  {len(test_split):,} rows, "
      f"{test_split['participant_id'].nunique()} participants")

# Reproducibility gate. If these do not match wk10, stop: something upstream
# has changed and nothing below is comparable to previously reported figures.
assert len(train_split) == 1_276_709, "train row count differs from wk10"
assert len(test_split)  ==   290_019, "test row count differs from wk10"
print("Row counts match wk10.")

README.md:   0%|          | 0.00/8.57k [00:00<?, ?B/s]

indoor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 29.8MB            

indoor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

indoor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.41MB            

indoor/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1566728 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/194829 [00:00<?, ? examples/s]

Train: 1,276,709 rows, 13 participants
Test:  290,019 rows, 3 participants
Row counts match wk10.


In [4]:
DROP_COLS = [
    "file_name", "Timestamp", "participant_id",
    "Air-Velocity", "Metabolic-Rate",
    "Nose", "Neck", "RShoulder", "RElbow",
    "LShoulder", "LElbow", "REye", "LEye", "REar", "LEar",
    "Emotion-Self", "Emotion-ML",
    "Label", "Label_3class",          # both targets, always dropped
]

def prepare_features(df, target_col):
    X = df.drop(columns=[c for c in DROP_COLS if c in df.columns],
                errors="ignore").copy()
    if "Gender" in X.columns:
        X["Gender"] = LabelEncoder().fit_transform(X["Gender"].astype(str))
    X = X.select_dtypes(include=[np.number])
    return X, df[target_col]

X_train_7, y_train_7 = prepare_features(train_split, "Label")
X_test_7,  y_test_7  = prepare_features(test_split,  "Label")
X_train_3, y_train_3 = prepare_features(train_split, "Label_3class")
X_test_3,  y_test_3  = prepare_features(test_split,  "Label_3class")

FEATURE_COLS = list(X_train_7.columns)

# Leakage gate. Cheap, and it fails loudly.
assert "Label" not in FEATURE_COLS and "Label_3class" not in FEATURE_COLS
assert len(FEATURE_COLS) == 18, f"expected 18 features, got {len(FEATURE_COLS)}"
print(f"{len(FEATURE_COLS)} features. Leakage assertions passed.")

18 features. Leakage assertions passed.


In [5]:
print("Baselines...")
for gran, Xtr, ytr, Xte, yte, want in [
    (7, X_train_7, y_train_7, X_test_7, y_test_7, 0.2858),
    (3, X_train_3, y_train_3, X_test_3, y_test_3, 0.7163),
]:
    t0 = time.time()
    clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                 n_jobs=-1).fit(Xtr, ytr)
    train_s = time.time() - t0
    t1 = time.time(); pred = clf.predict(Xte); infer_s = time.time() - t1
    r = log_result("Baseline (real only)", "TRTR", gran, yte, pred,
                   train_s, infer_s, len(Xte))
    status = "MATCH" if abs(r["macro_f1"] - want) < 0.001 else "*** MISMATCH ***"
    print(f"    vs wk10 {want:.4f} -> {status}")

BASE_7 = [r for r in RESULTS if r["granularity"] == 7][0]["macro_f1"]
BASE_3 = [r for r in RESULTS if r["granularity"] == 3][0]["macro_f1"]


Baselines...
  Baseline (real only)                   TRTR       7-class  macro_f1=0.2858  cold_f1=0.0000
    vs wk10 0.2858 -> MATCH
  Baseline (real only)                   TRTR       3-class  macro_f1=0.7163  cold_f1=0.7120
    vs wk10 0.7163 -> MATCH


In [6]:

print("Class-weighted Random Forest...")
for gran, Xtr, ytr, Xte, yte in [
    (7, X_train_7, y_train_7, X_test_7, y_test_7),
    (3, X_train_3, y_train_3, X_test_3, y_test_3),
]:
    t0 = time.time()
    clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                 class_weight="balanced", n_jobs=-1).fit(Xtr, ytr)
    train_s = time.time() - t0
    t1 = time.time(); pred = clf.predict(Xte); infer_s = time.time() - t1
    log_result("Class-weighted RF", "Cost-sensitive", gran, yte, pred,
               train_s, infer_s, len(Xte), notes="class_weight=balanced")
    if gran == 7:
        print(classification_report(yte, pred, zero_division=0))


Class-weighted Random Forest...
  Class-weighted RF                      Cost-sensitive 7-class  macro_f1=0.2504  cold_f1=0.0000
              precision    recall  f1-score   support

          -3       0.00      0.00      0.00     22556
          -2       0.03      0.03      0.03     19285
          -1       0.61      0.71      0.66     85260
           0       0.10      0.15      0.12     39044
           1       0.15      0.32      0.20     16824
           2       0.36      0.33      0.35     51786
           3       0.60      0.30      0.40     55264

    accuracy                           0.37    290019
   macro avg       0.26      0.26      0.25    290019
weighted avg       0.38      0.37      0.36    290019

  Class-weighted RF                      Cost-sensitive 3-class  macro_f1=0.6815  cold_f1=0.6567


In [7]:
from imblearn.ensemble import BalancedRandomForestClassifier

print("Balanced Random Forest...")
for gran, Xtr, ytr, Xte, yte in [
    (7, X_train_7, y_train_7, X_test_7, y_test_7),
    (3, X_train_3, y_train_3, X_test_3, y_test_3),
]:
    t0 = time.time()
    # These three arguments are set explicitly to match imblearn's future
    # defaults. Leaving them implicit emits a FutureWarning and, worse, means
    # a library upgrade would silently change your results.
    clf = BalancedRandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1,
        sampling_strategy="all", replacement=True, bootstrap=False,
    ).fit(Xtr, ytr)
    train_s = time.time() - t0
    t1 = time.time(); pred = clf.predict(Xte); infer_s = time.time() - t1
    log_result("Balanced RF", "Cost-sensitive", gran, yte, pred,
               train_s, infer_s, len(Xte),
               notes="sampling_strategy=all, replacement=True, bootstrap=False")
    if gran == 7:
        print(classification_report(yte, pred, zero_division=0))


Balanced Random Forest...
  Balanced RF                            Cost-sensitive 7-class  macro_f1=0.2429  cold_f1=0.0000
              precision    recall  f1-score   support

          -3       0.00      0.00      0.00     22556
          -2       0.06      0.08      0.07     19285
          -1       0.62      0.70      0.66     85260
           0       0.05      0.09      0.07     39044
           1       0.15      0.31      0.20     16824
           2       0.39      0.36      0.38     51786
           3       0.63      0.22      0.32     55264

    accuracy                           0.35    290019
   macro avg       0.27      0.25      0.24    290019
weighted avg       0.39      0.35      0.35    290019

  Balanced RF                            Cost-sensitive 3-class  macro_f1=0.6529  cold_f1=0.6656


In [8]:
# Save Part 1
pd.DataFrame(RESULTS).to_csv("results/wk12_costsensitive.csv", index=False)
from google.colab import files
files.download("results/wk12_costsensitive.csv")
print("Part 1 saved and downloaded.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Part 1 saved and downloaded.


 PART 2 — MOSTLYAI VARIANCE
#
# The generator exposes no seed. wk4 and wk10 disagree by roughly 0.04 macro
# F1 on the same input. Three further runs, pooled with wk10, give four draws
# and therefore a range you can report instead of a point estimate you cannot
# defend.

In [21]:
from mostlyai.sdk import MostlyAI

SDV_DROP = [
    "file_name", "Timestamp", "participant_id",
    "Air-Velocity", "Metabolic-Rate",
    "Nose", "Neck", "RShoulder", "RElbow",
    "LShoulder", "LElbow", "REye", "LEye", "REar", "LEar",
    "Emotion-Self", "Emotion-ML",
    "Label_3class",           # Label is KEPT: it is what we generate
]

mostly_train = train_split.drop(columns=SDV_DROP, errors="ignore").copy()
mostly_train["Gender"] = LabelEncoder().fit_transform(
    mostly_train["Gender"].astype(str))

SAMPLE_SIZE = 100_000
mostly_train_sample = (
    mostly_train.groupby("Label", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), int(SAMPLE_SIZE * len(x) / len(mostly_train))),
        random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

# Identical to wk10, so the runs pool legitimately.
assert len(mostly_train_sample) == 99_996, "sample size differs from wk10"
print(f"Training sample: {len(mostly_train_sample):,} rows (matches wk10)")



Training sample: 99,996 rows (matches wk10)


/tmp/ipykernel_5922/2747481497.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

In [22]:
def prep_synth(df, target_col):
    d = df.copy()
    if target_col == "Label_3class" and "Label_3class" not in d.columns:
        d["Label_3class"] = d["Label"].apply(
            lambda x: -1 if x <= -2 else (0 if x <= 1 else 1))
    if "Gender" in d.columns and d["Gender"].dtype == object:
        d["Gender"] = LabelEncoder().fit_transform(d["Gender"].astype(str))
    X = d.reindex(columns=FEATURE_COLS).apply(pd.to_numeric, errors="coerce")
    y = d[target_col]
    keep = X.notna().all(axis=1)
    return X[keep].reset_index(drop=True), y[keep].reset_index(drop=True)


def evaluate_synth(synth_df, tag):
    for gran, tcol, Xtr, ytr, Xte, yte in [
        (7, "Label",        X_train_7, y_train_7, X_test_7, y_test_7),
        (3, "Label_3class", X_train_3, y_train_3, X_test_3, y_test_3),
    ]:
        Xs, ys = prep_synth(synth_df, tcol)
        if ys.nunique() < 2:
            print(f"  [skip] {tag} {gran}-class: fewer than 2 classes")
            continue

        clf = RandomForestClassifier(n_estimators=100,
                                     random_state=RANDOM_STATE,
                                     n_jobs=-1).fit(Xs, ys)
        log_result(tag, "TSTR", gran, yte, clf.predict(Xte))

        Xa = pd.concat([Xtr.reset_index(drop=True), Xs], axis=0).reset_index(drop=True)
        ya = pd.concat([ytr.reset_index(drop=True), ys], axis=0).reset_index(drop=True)
        clf = RandomForestClassifier(n_estimators=100,
                                     random_state=RANDOM_STATE,
                                     n_jobs=-1).fit(Xa, ya)
        log_result(tag, "Augmented", gran, yte, clf.predict(Xte))


mostly = MostlyAI(local=True)

for run in [2, 3, 4]:          # run 1 is wk10
    print(f"\n=== MostlyAI training run {run} ===")
    t0 = time.time()
    g = mostly.train(name=f"autotherm_var_{RUN_ID}_r{run}",
                     data=mostly_train_sample)
    print(f"  trained in {time.time()-t0:.0f}s")
    synth = mostly.probe(g, size=len(mostly_train_sample))
    evaluate_synth(synth, f"MostlyAI run{run}")


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Initializing Synthetic Data SDK 6.1.1 in LOCAL mode 🏠

Connected to ]8;id=121759;file:///root/mostlyai\/root/]8;;\]8;id=990031;file:///root/mostlyai\mostlyai]8;;\ with 13 GB RAM, 2 CPUs, 0 GPUs available


=== MostlyAI training run 2 ===


Created generator 8789f6c7-5f85-4d3f-93dd-6762ecab0432

...

Created generator ae9bb43c-4e0e-4443-b697-ee808fefe3f7

Started generator training

Output()

🎉 Your generator is ready! Use it to create synthetic data. Publish it so others can do the same.

  trained in 1316s
  MostlyAI run4                          TSTR       7-class  macro_f1=0.2324  cold_f1=0.0000
  MostlyAI run4                          Augmented  7-class  macro_f1=0.2942  cold_f1=0.0000
  MostlyAI run4                          TSTR       3-class  macro_f1=0.7324  cold_f1=0.7480
  MostlyAI run4                          Augmented  3-class  macro_f1=0.7038  cold_f1=0.7347


In [23]:
res = pd.DataFrame(RESULTS)
res.to_csv("results/wk12_results.csv", index=False)

MANIFEST["finished"] = datetime.now().isoformat(timespec="seconds")
MANIFEST["n_conditions_logged"] = len(res)
with open("results/wk12_manifest.json", "w") as f:
    json.dump(MANIFEST, f, indent=2)

print(res.to_string(index=False))

# Variance summary. Pool these with the wk10 figures by hand: wk10 gave
# 3-class TSTR 0.6601 and Augmented 0.6719.
print("\n--- MostlyAI spread across runs 2 to 4 (this notebook only) ---")
m = res[res["method"].str.startswith("MostlyAI")]
for gran in [7, 3]:
    for proto in ["TSTR", "Augmented"]:
        v = m[(m.granularity == gran) & (m.protocol == proto)]["macro_f1"]
        if len(v):
            print(f"  {gran}-class {proto:<10} "
                  f"mean {v.mean():.4f}  min {v.min():.4f}  max {v.max():.4f}")

files.download("results/wk12_results.csv")
files.download("results/wk12_manifest.json")


              method       protocol  granularity  macro_f1  cold_f1  train_s  infer_ms_per_row          run_id                                                    notes
Baseline (real only)           TRTR            7    0.2858   0.0000    142.4            0.0084 20260805_150957                                                         
Baseline (real only)           TRTR            3    0.7163   0.7120    103.9            0.0039 20260805_150957                                                         
   Class-weighted RF Cost-sensitive            7    0.2504   0.0000    123.2            0.0071 20260805_150957                                    class_weight=balanced
   Class-weighted RF Cost-sensitive            3    0.6815   0.6567    105.4            0.0040 20260805_150957                                    class_weight=balanced
         Balanced RF Cost-sensitive            7    0.2429   0.0000     93.0            0.0066 20260805_150957 sampling_strategy=all, replacement=True, bootstra

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>